In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, col, expr, when, lit, window

def silver_ingestion(checkpoint_location_silver, checkpooint_location_malformed):
    # Read the bronze table
    try:
        bronze_df = spark.readStream.table("cosmetic_History.bronze.Users_Products_Info_Bronze")

        # Flatten the JSON string to separate columns
        bronze_transformed_df = bronze_df.withWatermark("event_time", "10 minutes") \
            .dropDuplicates(["user_id", "event_time", "event_type", "user_session", "product_info"]) \
            .withColumn('product_id', expr('product_info.product_id')) \
            .withColumn('category_id', expr('product_info.category_id')) \
            .withColumn('category_code', expr('product_info.category_code')) \
            .withColumn('brand', expr('product_info.brand')) \
            .withColumn('price', expr('product_info.price')) \
            .withColumn('Malformed_Records', when(
                (col('price') < 0) | (col('user_id').isNull()) | (col('event_time').isNull()),
                lit("Malformed")
            ).otherwise(lit("Valid"))).withColumn("window", window(col("event_time"), "10 minutes")) # Add 10-minute window for downstream aggregation
            

        # Route malformed records to separate table
        malformed_query = bronze_transformed_df.filter("Malformed_Records='Malformed'") \
            .writeStream \
            .format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", checkpooint_location_malformed) \
            .trigger(availableNow=True) \
            .table("cosmetic_History.silver.Malformed_Records")

        # Route valid records to Silver table including window_start
        silver_query = bronze_transformed_df.filter("Malformed_Records='Valid'") \
            .select("user_id", "event_type",
                    "product_id", "category_id", "category_code", "brand", "price",
                    "window") \
            .writeStream \
            .format("delta") \
            .outputMode("append") \
            .option("checkpointLocation", checkpoint_location_silver) \
            .trigger(availableNow=True) \
            .table("cosmetic_History.silver.Users_Products_Info_Silver")

        return {'malformed': malformed_query, 'silver': silver_query}
    except Exception as e:
        raise Exception(f"Error: {e}")


if __name__ == '__main__':
    print("Starting Silver Ingestion")
    checkpoint_location_silver = '/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/silver_checkpoint'
    checkpooint_location_malformed = "/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/silver_malformed_checkpoint"
    queries = silver_ingestion(checkpoint_location_silver, checkpooint_location_malformed)
    for q in queries.values():
        q.awaitTermination()
    print("Done")

Starting Silver Ingestion


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7028353733262434>, line 50
     48 checkpoint_location_silver = '/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/silver_checkpoint'
     49 checkpooint_location_malformed = "/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/silver_malformed_checkpoint"
---> 50 queries = silver_ingestion(checkpoint_location_silver, checkpooint_location_malformed)
     51 for q in queries.values():
     52     q.awaitTermination()

File <command-7028353733262434>, line 41, in silver_ingestion(checkpoint_location_silver, checkpooint_location_malformed)
     23 malformed_query = bronze_transformed_df.filter("Malformed_Records='Malformed'") \
     24     .writeStream \
     25     .format("delta") \
   (...)
     28     .trigger(availableNow=True) \
     29     .table("cosmetic_History.silver.Malformed_Records"